# Guardian Candidate Model v1 — First Production Training (Sprint 20)

Runs the exact Sprint 20 spec end to end on a Colab **T4 GPU**, using the official Apache-2.0 YOLOX integration from Sprint 19.1 (`OfficialYoloxTrainer` — nothing here knows YOLOX's internals):

**Dataset → Official YOLOX Training → Evaluate → Error Analysis → Qualitative Report → ONNX Export → Benchmark → COCO-Pretrained Comparison → Candidate**

Rules that still apply here:
- Data comes **only** from the Guardian Dataset Registry (`guardian-fall-detection-v1@1.0.0`, published in Sprint 18) — never raw datasets, never modified.
- `python -m guardian_ai.train` is the same CLI used locally; this notebook adds no training logic of its own. No architecture changes, no detector changes — Sprint 19.1's `DetectorFamily` wrapping is used as-is.
- This produces a **candidate model only**. Nothing here installs into any model zoo or touches the Edge Box — promotion is a separate, later, human decision.

> Hardware: Colab → Runtime → Change runtime type → **T4 GPU**.

## 1. Mount Google Drive

Drive holds a copy of the **published** `guardian-fall-detection-v1` registry directory (from Sprint 18 — `datasets/registry/guardian-fall-detection-v1/`, ~9GB) since it is too large to regenerate on Colab. Upload it once to `MyDrive/guardian-ai-data/registry/guardian-fall-detection-v1/` before running this cell.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DATA = "/content/drive/MyDrive/guardian-ai-data"
DRIVE_OUT = "/content/drive/MyDrive/guardian-ai-models/model-v1"
!mkdir -p {DRIVE_OUT}

## 2. Clone the repo and install dependencies

Cloning the full repo (not just `ai/`) so the COCO-pretrained comparison step can reuse the existing, already-reviewed `edge/guardian_edge/tools/install_yolox.py` installer — a hash-pinned Apache-2.0 artifact fetch (ADR-0003), not a new download this notebook invents.

The clone is guarded (`if not os.path.exists(...)`) so re-running this cell after a Colab disconnect/reconnect (see the resume note after training, below) does not fail on an already-cloned directory.

In [ ]:
import os

# Clone repository automatically
if not os.path.exists("/content/guardian-ai"):
    !git clone https://github.com/Xaliljon/guardian-ai.git

%cd /content/guardian-ai

# Install project
!pip install -e ai

## 3. Copy the published dataset from Drive

This is the real, checksummed Sprint 18 registry export (357 videos, 24,901 boxes) — copied in, never regenerated, so `VideoDatasetRegistry.get()` verifies the exact same bytes every run.

In [ ]:
!mkdir -p datasets/registry
!cp -r {DRIVE_DATA}/registry/guardian-fall-detection-v1 datasets/registry/
!python -c "\
from pathlib import Path; \
from guardian_ai.acquisition.registry import VideoDatasetRegistry; \
print(VideoDatasetRegistry(Path('datasets/registry'))\
    .get('guardian-fall-detection-v1', '1.0.0'))"

## 4. Train — the exact Sprint 20 hyperparameters

`ai/training/configs/model-v1.yaml`: official YOLOX-Tiny (Apache-2.0, Sprint 19.1), 640px, COCO-pretrained checkpoint (auto-downloaded, checksum-pinned), 30 epochs, batch 16, workers 2, seed 42, SGD lr=0.01 with **5-epoch linear warmup then cosine annealing** (fixes the numerical divergence Sprint 19.1's comparison report disclosed), early stopping patience 10, mixed precision on, `device: cuda` (the T4). One command, one YAML — the config is the complete, reproducible recipe.

In [ ]:
!python -m guardian_ai.train train --config ai/training/configs/model-v1.yaml

import pathlib

RUN = sorted(pathlib.Path("ai/training/runs").iterdir())[-1]
print("run:", RUN)

If Colab disconnects mid-training, reconnect, re-run cells 1–3, then resume from the last checkpoint (optimizer/scheduler/epoch/early-stopping state all restored):

```
!python -m guardian_ai.train resume --run {RUN}
```

## 5. Evaluate + visual reports

The full mandated metric set — precision, recall, F1, mAP@50, mAP@50-95, per-class, confusion matrix, absolute FP/FN — plus confusion matrix / PR curve / loss curve / summary PDF.

In [ ]:
!python -m guardian_ai.train evaluate --run {RUN}
!python -m guardian_ai.train report --run {RUN}

from IPython.display import Image as ShowImage
from IPython.display import display

display(ShowImage(filename=f"{RUN}/reports/loss_curve.png"))
display(ShowImage(filename=f"{RUN}/reports/confusion_matrix.png"))

## 6. Error analysis

Top false-positive images, top false-negative images, the most confident wrong detections, the worst-localized correct detections, and the most confused class pairs — written to `reports/error-analysis.json`.

In [ ]:
!python -m guardian_ai.train error-analysis --run {RUN} --top-k 10

## 7. Qualitative report

50 random validation predictions, ground truth (green) and predictions (cyan, with confidence) drawn on the same letterboxed frame the model saw.

In [ ]:
!python -m guardian_ai.train qualitative --run {RUN} --count 50

import glob

samples = sorted(glob.glob(f"{RUN}/reports/qualitative/*.png"))[:4]
for path in samples:
    display(ShowImage(filename=path))

## 8. ONNX export (validated) + benchmark

Export runs the ONNX structural checker **and** a torch-vs-onnxruntime parity check (max |Δ| ≤ 1e-4) — a diverging artifact is rejected before any manifest exists. Benchmark measures latency/memory/size on ONNX Runtime CPU — these numbers become the baseline for every future run.

In [ ]:
!python -m guardian_ai.train export --run {RUN} --version 1.0.0 --name guardian-fall-v1
!python -m guardian_ai.train benchmark --run {RUN} --runs 50

## 9. COCO-pretrained comparison — PROMOTE or KEEP COCO

Fetches the official Apache-2.0 YOLOX-Tiny COCO checkpoint (pinned SHA-256, ADR-0003) through the existing edge/ installer, evaluates it on the *same* test images with our own harness (person class only), and compares precision/recall/false-positives/latency/memory against this run.

In [ ]:
!python -m guardian_ai.train coco-compare --run {RUN} \
    --zoo-root /content/coco-baseline-zoo \
    --edge-project-root /content/guardian-ai/edge \
    --runs 50

import json

comparison = json.loads((RUN / "coco-comparison.json").read_text())
print("VERDICT:", comparison["verdict"])
for reason in comparison["reasons"]:
    print(" -", reason)

## 10. Mark as candidate — never deploy from here

This is a **candidate model only**. It is never installed into any model zoo from this notebook; promotion is a separate, later, human decision made after reviewing everything above (see `reports/model-v1/README.md` in the repo for the write-up template).

In [ ]:
!python -m guardian_ai.train candidate --run {RUN} \
    --notes "Full Colab T4 run: 19,940 train images, 30 epochs, COCO-pretrained, \
5-epoch warmup + cosine, mixed precision"

!cp {RUN}/export/model.onnx {RUN}/export/manifest.json \
    {RUN}/export/guardian-validation.json {DRIVE_OUT}/
!cp {RUN}/reports/evaluation.json {RUN}/reports/error-analysis.json \
    {RUN}/coco-comparison.json {DRIVE_OUT}/
print("Artifacts + reports copied to", DRIVE_OUT)
print("Candidate only. No zoo install.")
print("Hand this run directory to review for the promotion decision.")